In [1]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime
import plotly.express as px

Set variables

In [2]:
current_path = os.getcwd() 

Import Data

In [3]:
df_appointments = pd.read_parquet(os.path.join(current_path, "bucket", "landing", "appointments.parquet"))
df_patients = pd.read_parquet(os.path.join(current_path, "bucket", "landing", "patients.parquet"))
df_prescriptions = pd.read_parquet(os.path.join(current_path, "bucket", "landing", "prescriptions.parquet"))
df_providers = pd.read_parquet(os.path.join(current_path, "bucket", "landing", "providers.parquet"))


In [4]:
# Garante que 'appointment_date' é datetime
df_appointments['appointment_date'] = pd.to_datetime(df_appointments['appointment_date'])

# 1. Dia da semana (Monday, Tuesday, etc.)
df_appointments['day_of_week'] = df_appointments['appointment_date'].dt.day_name()

# 2. Dias desde o último appointment

# Ordena por patient_id e appointment_date para ficar certo
df_appointments = df_appointments.sort_values(['patient_id', 'appointment_date'])

# Calcula a diferença de dias entre appointments por paciente
df_appointments['days_since_last_appointment'] = df_appointments.groupby('patient_id')['appointment_date'].diff().dt.days

# Se quiser, pode preencher o primeiro agendamento como -1 ou outro valor
df_appointments['days_since_last_appointment'] = df_appointments['days_since_last_appointment'].fillna(-1).astype(int)


In [5]:
df_appointments 

,appointment_id,patient_id,appointment_date,appointment_type,provider_id,day_of_week,days_since_last_appointment
3,4,1,2023-01-16,Consultation,6,Monday,-1
74,75,1,2023-03-18,Checkup,5,Saturday,61
75,76,1,2023-04-04,Consultation,2,Tuesday,17
59,60,2,2023-01-06,Emergency,4,Friday,-1
93,94,2,2023-01-12,Consultation,9,Thursday,6
...,...,...,...,...,...,...,...
100,101,51,2022-12-15,Checkup,4,Thursday,-1
101,102,52,2023-01-20,Consultation,2,Friday,-1
102,103,53,2023-01-10,Emergency,7,Tuesday,-1
103,104,54,2023-02-05,Checkup,5,Sunday,-1


In [6]:
df_patients['age_group'] = (
    pd.cut(
        df_patients['age'],
        bins=[-1, 18, 30, 50, 70, float('inf')],
        labels=['0-18', '19-30', '31-50', '51-70', '71+']
    )
    .astype(str)
    .replace('nan', 'Unknown') 
)

# Registration
df_patients['registration_date'] = pd.to_datetime(df_patients['registration_date'])
today = pd.Timestamp.today()


def calculate_months_since(date):
    return (today.year - date.year) * 12 + (today.month - date.month)

df_patients['months_registered'] = df_patients['registration_date'].apply(calculate_months_since)



# 3. Cria a coluna patient_type
df_patients['patient_type'] = pd.cut(
    df_patients['months_registered'],
    bins=[-1, 6, 24, float('inf')],
    labels=['New', 'Regular', 'Long-term']
)

# 4. (Opcional) tratar nulos
df_patients['patient_type'] = df_patients['patient_type'].cat.add_categories('Unknown')
df_patients['patient_type'] = df_patients['patient_type'].fillna('Unknown')


In [7]:

df_prescriptions['prescription_date'] = pd.to_datetime(df_prescriptions['prescription_date'])

# 1. Ordena
df_prescriptions = df_prescriptions.sort_values(['patient_id', 'medication_name', 'prescription_date'])

# 2. Calcula a diferença de dias (mantendo NaN nas primeiras prescrições)
df_prescriptions['prescription_frequency'] = df_prescriptions.groupby(
    ['patient_id', 'medication_name']
)['prescription_date'].diff().dt.days

# 3. Calcula a média da frequência (enquanto ainda tem NaN) 
df_prescriptions['avg_prescription_frequency'] = df_prescriptions.groupby(
    ['patient_id', 'medication_name']
)['prescription_frequency'].transform('mean').round(1)

# 4. Calcula o número de repetições (contagem de prescrições)
df_prescriptions['prescription_repeats'] = df_prescriptions.groupby(
    ['patient_id', 'medication_name']
)['prescription_id'].transform('count')

# 5. Agora sim: preenche NaN em prescription_frequency com -1
df_prescriptions['prescription_frequency'] = df_prescriptions['prescription_frequency'].fillna(-1).astype(int)

# Resultado


In [ ]:
df_prescriptions.sort_values("prescription_repeats", ascending=False).head(10)           

,prescription_id,patient_id,medication_name,prescription_date,prescription_frequency,avg_prescription_frequency,prescription_repeats
127,128,22,Aspirin,2023-04-04,1,18.7,4
126,127,22,Aspirin,2023-04-03,1,18.7,4
28,29,22,Aspirin,2023-04-02,54,18.7,4
14,15,22,Aspirin,2023-02-07,-1,18.7,4
4,5,24,Atorvastatin,2023-03-29,29,15.0,3
10,11,40,Aspirin,2023-02-10,21,35.5,3
92,93,24,Atorvastatin,2023-02-28,-1,15.0,3
24,25,12,Aspirin,2023-01-07,-1,42.0,3
123,124,8,Atorvastatin,2023-01-26,1,1.0,3
122,123,8,Atorvastatin,2023-01-25,1,1.0,3


In [9]:
df_prescriptions[df_prescriptions["patient_id"]==22]

,prescription_id,patient_id,medication_name,prescription_date,prescription_frequency,avg_prescription_frequency,prescription_repeats
14,15,22,Aspirin,2023-02-07,-1,18.7,4
28,29,22,Aspirin,2023-04-02,54,18.7,4
126,127,22,Aspirin,2023-04-03,1,18.7,4
127,128,22,Aspirin,2023-04-04,1,18.7,4
81,82,22,Atorvastatin,2023-02-16,-1,NaN,1
67,68,22,Lisinopril,2023-02-06,-1,NaN,1



What is the distribution of patients across age groups?

In [10]:
df_patients['age_group'].value_counts()

age_group
71+        15
31-50      14
51-70      12
19-30       8
Unknown     5
0-18        1
Name: count, dtype: int64


How does the appointment frequency vary by patient type?

In [11]:

df1 = df_appointments.merge(
    df_patients,
    on='patient_id',
    how='left'
)

df1 = df1[df1['days_since_last_appointment'] >= 0]


analysis = df1.groupby('patient_type')['days_since_last_appointment'].agg(['count', 'mean', 'median', 'std']).round(2)


/var/folders/qs/t5dq2fb92vx3_q20m013swcm0000gn/T/ipykernel_18139/1263341116.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  analysis = df1.groupby('patient_type')['days_since_last_appointment'].agg(['count', 'mean', 'median', 'std']).round(2)


In [12]:
analysis

,count,mean,median,std
patient_type,,,,
New,2,9.50,9.5,2.12
Regular,37,25.65,17.0,21.42
Long-term,18,23.11,10.0,25.77
Unknown,0,NaN,NaN,NaN


What are the most common appointment types by age group?


In [13]:
# Primeiro: agrupar e contar normalmente (como já fizemos)
appointment_counts = df1.groupby(['age_group', 'appointment_type']).size().reset_index(name='count')

# Agora: para cada grupo de idade, manter apenas os tipos mais frequentes
# 1. Calcula o maior número de consultas para cada faixa etária
max_counts = appointment_counts.groupby('age_group')['count'].transform('max')

# 2. Filtra apenas as linhas que têm o count máximo
top_appointment_counts = appointment_counts[appointment_counts['count'] == max_counts]

# Organiza bonitinho
top_appointment_counts = top_appointment_counts.sort_values(['age_group', 'appointment_type']).reset_index(drop=True)


In [16]:
fig = px.bar(
    appointment_counts,
    x='age_group',
    y='count',
    color='appointment_type',
    barmode='group',
    title='Tipos de Consulta Mais Comuns por Faixa Etária',
    labels={
        'age_group': 'Faixa Etária',
        'count': 'Número de Consultas',
        'appointment_type': 'Tipo de Consulta'
    },
    width=900,
    height=500
)

fig.show()

In [14]:
top_appointment_counts

,age_group,appointment_type,count
0,0-18,Consultation,1
1,19-30,Checkup,4
2,19-30,Consultation,4
3,19-30,Emergency,4
4,31-50,Consultation,5
5,31-50,Emergency,5
6,51-70,Checkup,7
7,51-70,Emergency,7
8,71+,Checkup,9


How does prescription frequency correlate with appointment frequency?

In [ ]:
patient_appointments = df1.groupby('patient_id').size().reset_index(name='num_appointments')

# 2. Número de prescrições por paciente
patient_prescriptions = df_prescriptions.groupby('patient_id').size().reset_index(name='num_prescriptions')

# 3. Juntar as duas tabelas
patient_activity = patient_appointments.merge(
    patient_prescriptions,
    on='patient_id',
    how='inner'  
)



In [29]:
patient_activity

,patient_id,num_appointments,num_prescriptions
0,1,2,6
1,2,1,3
2,6,1,4
3,7,2,1
4,8,2,7
5,10,1,3
6,14,2,1
7,16,1,1
8,17,2,1
9,18,2,1


In [30]:
correlation = patient_activity['num_appointments'].corr(patient_activity['num_prescriptions'])




In [34]:
correlation
print(f"Correlação entre número de consultas e número de prescrições: {correlation:.2f}")


Correlação entre número de consultas e número de prescrições: -0.13
